# Autoencoder Foundations & Denoising

## What are Autoencoders?

An **autoencoder** is an unsupervised neural network that learns to compress data into a lower-dimensional representation (encoding) and then reconstruct the original data from this compressed form (decoding).

### Architecture Components

```
Input → [ENCODER] → Bottleneck (Latent Space) → [DECODER] → Reconstruction
  x    →    f(x)   →        z               →    g(z)    →      x̂
```

**Three main components:**

1. **Encoder** (f): Compresses input x into latent representation z
   - Maps high-dimensional input to low-dimensional code
   - Learns important features while discarding noise

2. **Bottleneck/Latent Space** (z): Compressed representation
   - Forces network to learn efficient encoding
   - Dimensionality determines compression level
   - Contains learned features of the data

3. **Decoder** (g): Reconstructs input from latent representation
   - Maps low-dimensional code back to original space
   - Learns to generate data from compressed features

### Mathematical Formulation

**Objective:** Minimize reconstruction error

$$
\mathcal{L}(x, \hat{x}) = \|x - \hat{x}\|^2 = \|x - g(f(x))\|^2
$$

Where:
- $x$ = original input
- $f(x) = z$ = encoder output (latent representation)
- $g(z) = \hat{x}$ = decoder output (reconstruction)
- $\mathcal{L}$ = loss function (typically MSE or BCE)

**Training Process:**
1. Forward pass: $\hat{x} = g(f(x))$
2. Compute loss: $\mathcal{L}(x, \hat{x})$
3. Backpropagate gradients
4. Update weights to minimize reconstruction error

### Key Characteristics

**Unsupervised Learning:**
- No labels required
- Learns from data structure itself
- Self-supervised: input = target

**Dimensionality Reduction:**
- Bottleneck forces compression
- Similar to PCA but non-linear
- Learns task-specific representations

**Applications:**
- **Denoising**: Remove noise from corrupted data
- **Compression**: Reduce data size for storage/transmission
- **Anomaly Detection**: High reconstruction error for unusual patterns
- **Feature Learning**: Use encoder for downstream tasks
- **Data Generation**: Sample from latent space and decode

### Comparison with PCA

| Aspect | PCA | Autoencoder |
|--------|-----|-------------|
| Linearity | Linear transformations only | Non-linear (with activation functions) |
| Optimization | Closed-form solution | Iterative gradient descent |
| Flexibility | Fixed orthogonal components | Learned representations |
| Complexity | Fast, simple | Slower, more complex |
| Expressiveness | Limited to linear relationships | Can capture complex patterns |

## 1. Core Theory: The Information Bottleneck

Autoencoder is a type of neural network that is trained to attempt to copy its input to its output.

### Architecture Components:
1. **Encoder ($f$):** Maps the input $x$ to a compressed representation $h = f(x)$.
2. **Bottleneck:** A layer with significantly fewer neurons than the input, forcing the network to learn a compressed "latent" code.
3. **Decoder ($g$):** Maps the latent code back to the original dimension $\hat{x} = g(h)$.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Helper Function for Visualization
We'll use a simple function to compare original, noisy, and cleaned images side-by-side.

In [ ]:

def visualize_denoising(original, noisy, reconstructed, n=8):
    """
    Visualize denoising results.
    Original: Clean images
    Noisy: Noisy images
    Reconstructed: Denoised images
    n: Number of images to display
    """
    
    plt.figure(figsize=(15, 6))
    for i in range(n):
        # Original
        ax = plt.subplot(3, n, i + 1)
        plt.imshow(original[i].cpu().squeeze(), cmap='gray')
        plt.axis('off')
        if i == 0: ax.set_title('Clean Original')

        # Noisy
        ax = plt.subplot(3, n, i + 1 + n)
        plt.imshow(noisy[i].cpu().squeeze(), cmap='gray')
        plt.axis('off')
        if i == 0: ax.set_title('Noisy Input')

        # Reconstructed
        ax = plt.subplot(3, n, i + 1 + 2*n)
        plt.imshow(reconstructed[i].cpu().squeeze(), cmap='gray')
        plt.axis('off')
        if i == 0: ax.set_title('Denoised Output')
    plt.tight_layout()
    plt.show()


## 2. Dataset Setup: MNIST
We will use the standard MNIST dataset of handwritten digits.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 3. Defining the Autoencoder
We'll use a simple Linear (Fully-Connected) architecture. Notice the bottleneck at `latent_dim=32`.

In [ ]:
class SimpleAE(nn.Module):
    def __init__(self, latent_dim=32):
        super(SimpleAE, self).__init__()
        # Encoder: 784 -> 128 -> 64 -> 32
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        # Decoder: 32 -> 64 -> 128 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 28*28),
            nn.Sigmoid(), # Output pixels in range [0, 1]
            nn.Unflatten(1, (1, 28, 28))
        )

    def forward(self, x):
        h = self.encoder(x)
        x_hat = self.decoder(h)
        return x_hat

model = SimpleAE(latent_dim=32).to(device)
print(model)


## 4. Training for Denoising
Crucially, we add noise to the input inside the training loop, but the target remains the *clean* image.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def add_noise(img, factor=0.4):
    noise = torch.randn_like(img) * factor
    return torch.clamp(img + noise, 0., 1.)

epochs = 10
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for images, _ in train_loader:
        images = images.to(device)
        noisy = add_noise(images)
        
        outputs = model(noisy) # Noise image
        loss = criterion(outputs, images) # Target is CLEAN image
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}")


## 5. Visualizing Denoising Performance
Let's see how our trained model performs on unseen noisy images from the test set.

In [ ]:

model.eval()
with torch.no_grad():
    data_iter = iter(test_loader)
    images, _ = next(data_iter)
    images = images.to(device)
    noisy = add_noise(images)
    reconstructed = model(noisy)
    
    visualize_denoising(images, noisy, reconstructed)


## 6. Understanding the Latent Space
What does the bottleneck actually look like? If we use a `latent_dim=2`, we can plot the network's internal 'map' of the digits.

In [ ]:

# Train a 2D model briefly
model_2d = SimpleAE(latent_dim=2).to(device)
opt_2d = optim.Adam(model_2d.parameters(), lr=0.001)

# Tiny training loop (3 epochs) to show clustering
for epoch in range(3):
    for images, _ in train_loader:
        images = images.to(device)
        loss = criterion(model_2d(images), images)
        opt_2d.zero_grad() 
        loss.backward()
        opt_2d.step()

# Plot the 2D Latent Space
model_2d.eval()
latents, targets = [], []
with torch.no_grad():
    for images, labels in test_loader:
        z = model_2d.encoder(images.to(device))
        latents.append(z.cpu().numpy())
        targets.append(labels.numpy())

latents = np.vstack(latents)
targets = np.concatenate(targets)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(latents[:, 0], latents[:, 1], c=targets, cmap='tab10', alpha=0.5)
plt.colorbar(scatter)
plt.title("2D Latent Space Map (Bottleneck)")
plt.xlabel("Latent Feature 1")
plt.ylabel("Latent Feature 2")
plt.show()


## 7. Moving Beyond Linear Layers: Convolutional AE

While linear layers work for small MNIST images, real-world images possess spatial structure. 

**Key Idea:** Instead of `Linear`, we use `Conv2d` to compress and `ConvTranspose2d` to upsample. 

#### The Architecture Pattern:
1. **Encoder:** Input $\to$ Conv $\to$ MaxPool $\to$ Conv $\to$ Bottleneck
2. **Decoder:** Bottleneck $\to$ ConvTranspose (or Upsample+Conv) $\to$ Output

**Challenge:** Can you think of how you would change the `SimpleAE` class above to use `nn.Conv2d` layers? (Note: Keep track of image dimensions $28 \to 14 \to 7 \to \dots$)

## 8. Workshop Exercises

1. **Latent Squeeze:** Change the bottleneck to just `latent_dim=1`. How much detail is lost? Can the model still distinguish any digits?
2. **The Noise Battle:** Increase the `noise_factor` to 0.8. Does the model still produce clean digits, or does it start failing?
3. **Fashion Shift:** Try running this exact notebook but replace `datasets.MNIST` with `datasets.FashionMNIST`. What changes will be there in the reconstruction quality?